
## MLを用いた次元圧縮による説明変数空間図示

### データ作成からデータ解析 

炭素構造探索結果の説明変数をMLで変換し次元圧縮して視覚的に意味のある変換になるのかを確かめる。


In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)


In [ ]:
from sklearn.manifold import MDS
from sklearn.manifold import TSNE

# from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler 

In [ ]:
g_ml_name = "TSNE" # MDS, TSNE

In [ ]:

def get_data(data_name):
    """データ取得

    Args:
        data_name (str): データ名

    Raises:
        ValueError: 規定外データ名の場合

    Returns:
        pd.DataFrame: データ。
        [str]: 説明変数名リスト。
        [str]: 目的変数名リスト。
        [str]: メタ変数名リスト。
    """
    if data_name == "ReCo":
        df = pd.read_csv("../data/TC_ReCo_detail_descriptor.csv")
        descriptor_names = ['C_R', 'C_T', 'vol_per_atom', 'Z', 'f4', 'd5', 'L4f', 'S4f', 'J4f',
                             '(g-1)J4f', '(2-g)J4f']
        target_name = 'Tc'
        mata_labels = ['name', 'polytyp', 'ref',
                       'author', 'link', 'comment', 'polytyp2']
    elif data_name=="ZBWZ":
        df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
        descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
                             'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
                             'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
        target_name = 'dE'
        meta_names = None

        # dE が特に大きい　C C を除く
        i = np.argmax(df["dE"].values)
        df.drop(index=i, inplace=True)
    else:
        raise ValueError("unknown inputfile={}".format(inputfile))
        
    return df, descriptor_names, target_name, mata_labels

g_data_name = "ReCo"
g_df, g_descriptor_names, g_target_name, g_mata_labels = get_data(g_data_name)

In [ ]:
def apply_normalize_mds(df, descriptor_names, ml_name="MDS", ndim=2):
    """MDSで次元圧縮を行う。

    Args:
        df (pd.DataFrame): データ。
        descriptor_names ([str]): 説明変数。
        ndim (int, optional): 変換次元. Defaults to 2.

    Returns:
        np.ndarray: 変換された説明変数。
    """
    Xraw = df.loc[:, descriptor_names].values
    # プリプロセス（の一部）
    # データ規格化
    scaler = StandardScaler()
    scaler.fit(Xraw)
    X = scaler.transform(Xraw)
    # データ解析
    if ml_name =="MDS":
        ml = MDS(ndim)
    elif ml_name == "TSNE":
        ml = TSNE(ndim)
    else:
        raise ValueError(f"unknown ml_name={ml_name}.")
    X_mds = ml.fit_transform(X)
    return X_mds

g_X_mds = apply_normalize_mds(g_df, g_descriptor_names, ml_name= g_ml_name)
g_y = g_df.loc[:, g_target_name].values

In [ ]:
g_ml_name

### 図示 

In [ ]:
def plot_X_y(X_pca, y):
    """説明変数の二次元での図示。

    Args:
        X_pca (np.ndarray): 説明変数
        y (np.ndarray): 目的変数
    """    
    fig, ax = plt.subplots()
    cm = plt.get_cmap("rainbow")
    mappable = ax.scatter(X_pca[:, 0], X_pca[:, 1], marker=".", c=y, cmap=cm)
    ax.set_xlabel("ML1")
    ax.set_ylabel("ML2")
    fig.colorbar(mappable, ax=ax)
    
plot_X_y(g_X_mds, g_y)

In [ ]:
g_X_mds = apply_normalize_mds(g_df, g_descriptor_names, ndim=3)
g_y = g_df.loc[:, g_target_name].values

回転できる３D表示を行う。（場合によってはできない。）

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
# %matplotlib notebook
def show_3d(X,y):
    """説明変数の3次元での図示。

    Args:
        X (np.ndarray): 説明変数
        y (np.ndarray): 目的変数。
    """
    angle = 30
    fig = plt.figure(figsize=(5,5))
    ax = fig.add_subplot(111, projection="3d")
    cm = plt.get_cmap("rainbow")
    im = ax.scatter(X[:, 0], X[:, 1],
                    X[:, 2], marker=".", c=y, cmap=cm)
    ax.view_init(30, angle)
    ax.set_title("angle={}".format(angle))
    ax.set_xlabel("ML1")
    ax.set_ylabel("ML2")
    ax.set_zlabel("ML3")
    fig.colorbar(im)
show_3d(g_X_mds, g_y)

# 付録

ML軸で相間と回帰を行い性能指標でどちらが良い軸なのかを評価する。

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from scipy.stats import pearsonr, spearmanr

def show_pearsonr(X_pca,y):
    """Pearsonの相関関数を計算する。

    Args:
        X_pca (np.ndarray): 説明変数
        y (np.ndarray): 目的変数
    """    
    for i in range(3):
        pr, _ = pearsonr(X_pca[:, i], y)
        sr, _ = spearmanr(X_pca[:, i], y)
        print(i, "R {}, {}".format(pr, sr))

show_pearsonr(g_X_mds, g_y)

In [ ]:
def linear_fit(X, y):
    """evaluate sore in the linear regression

    Args:
        X (np.array): explanatory variables.
        y (np.array): target variable.
    """        
    reg = LinearRegression(fit_intercept=True)
    reg.fit(X, y)
    yp = reg.predict(X)
    score = reg.score(X, y)
    print("R2=", score)


g_xlist = [0, 1]
print("descriptor", g_xlist)
linear_fit(g_X_mds[:, g_xlist], g_y)

g_xlist = [1, 2]
print("descriptor", g_xlist)
linear_fit(g_X_mds[:, g_xlist], g_y)


g_xlist = [0, 2]
print("descriptor", g_xlist)
linear_fit(g_X_mds[:, g_xlist], g_y)

一つだけ取ると0,2,1の順。
２つ取ると[1,2]、[0,1]の順に目的変数に対して良い説明変数になっていることが分かる。

## 参考

大量のデータに対して高速に変換でき、少数ならば妥当にtransformが別途可能な
UMAP (Uniform Manifold Approximation and Projection）が最近は人気です。

ref. https://umap-learn.readthedocs.io/en/latest/index.html

## 問題

上のlinear_fitはモデル当てはめの場合の回帰性能である。
予測性能を評価せよ。

## 問題

別のデータを用いて実行せよ。